In [ ]:
import torch
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("../")
DATASET_PATH = PROJECT_ROOT / "datasets" / "depth_v1"

In [ ]:
from torch.utils.data import Dataset

class DepthHandDataset(Dataset):
    def __init__(self, npz_paths, frames_per_clip=60):
        self.npz_paths = list(npz_paths)
        self.frames_per_clip = frames_per_clip
        self.total_frames = len(self.npz_paths) * self.frames_per_clip
        
    def __len__(self):
        return self.total_frames
    
    def __getitem__(self, idx):
        clip_idx = idx // self.frames_per_clip
        frame_idx = idx % self.frames_per_clip
        
        clip_path = self.npz_paths[clip_idx]
        
        with np.load(clip_path) as data:
            depth = data["depth"][frame_idx]
            joints = data["joints_3d"][frame_idx]
            
            x = torch.from_numpy(depth).float().unsqueeze(0)
            y = torch.from_numpy(joints).float() 
            return x, y

In [ ]:
import random
from torch.utils.data import DataLoader
SPLIT_RATIO = 0.9

random.seed(42)
npz_clips = sorted(list(DATASET_PATH.glob("*.npz")))
clips_shuffled = npz_clips.copy()
random.shuffle(clips_shuffled)

split_idx = int(SPLIT_RATIO * len(clips_shuffled))
train_files = clips_shuffled[:split_idx]
val_files = clips_shuffled[split_idx:]

print(f"Clips for TRAIN: {len(train_files)} ({len(train_files)*60} frames)")
print(f"Clips for VAL:   {len(val_files)} ({len(val_files)*60} frames)")

train_dataset = DepthHandDataset(train_files)
val_dataset = DepthHandDataset(val_files)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class HandPoseIPR(nn.Module):
    def __init__(self, num_joints: int = 21):
        super().__init__()
        self.num_joints = num_joints
        backbone = models.resnet18(weights=None)

        # 1st conv - 1 channel depth
        old_conv = backbone.conv1
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # deconv head
        self.deconv_head = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, num_joints * 2, kernel_size=1)
        )

        self.coord_scale = nn.Parameter(torch.tensor([0.18, 0.18, 0.12]))
        self.coord_bias = nn.Parameter(torch.tensor([0.00, 0.00, 0.22]))

    def forward(self, x):
        # x: (Batch, 1, 480, 848)

        x_feat = self.conv1(x)
        x_feat = self.bn1(x_feat)
        x_feat = self.relu(x_feat)
        x_feat = self.maxpool(x_feat)

        x_feat = self.layer1(x_feat)
        x_feat = self.layer2(x_feat)
        x_feat = self.layer3(x_feat)
        x_feat = self.layer4(x_feat)  # (Batch, 512, 15, 27)

        out = self.deconv_head(x_feat)  # (Batch, 42, 60, 108)

        heatmaps = out[:, :self.num_joints]   # (Batch, 21, 60, 108)
        depthmaps = out[:, self.num_joints:]  # (Batch, 21, 60, 108)

        B, K, H, W = heatmaps.shape
        prob = F.softmax(heatmaps.view(B, K, -1), dim=-1).view(B, K, H, W)

        grid_x = torch.linspace(-1.0, 1.0, W, device=x.device).view(1, 1, 1, W)
        grid_y = torch.linspace(-1.0, 1.0, H, device=x.device).view(1, 1, H, 1)

        pred_x = (prob * grid_x).sum(dim=(-2, -1))                 # (Batch, 21)
        pred_y = (prob * grid_y).sum(dim=(-2, -1))                 # (Batch, 21)
        pred_z = (prob * torch.tanh(depthmaps)).sum(dim=(-2, -1))  # (Batch, 21)

        norm_joints = torch.stack([pred_x, pred_y, pred_z], dim=-1)  # (Batch, 21, 3)
        joints_3d = norm_joints * self.coord_scale + self.coord_bias  # (Batch, 21, 3)

        return joints_3d

In [ ]:
def mpjpe(y_hat, y_gt):
    diff = y_hat - y_gt # (Batch, 21, 3)
    dist = torch.linalg.vector_norm(diff, dim=-1) 
    
    return dist.mean() * 1000 # mjpe in mm
 
criterion = nn.MSELoss()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = HandPoseIPR().to(device)

In [ ]:
from tqdm import tqdm

EPOCHS = 10

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val_mpjpe = float("inf")

for epoch in range(EPOCHS):
    # TRAIN
    net.train()
    train_loss_total = 0.0
    train_mpjpe_total = 0.0

    train_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for x, y in train_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        y_hat = net(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item()
        with torch.no_grad():
            batch_mpjpe = mpjpe(y_hat, y).item()
            train_mpjpe_total += batch_mpjpe

        train_bar.set_postfix({
            "loss": f"{loss.item():.5f}",
            "mpjpe": f"{batch_mpjpe:.1f} mm"
        })

    train_bar.close()   
    train_loss_avg = train_loss_total / len(train_dataloader)
    train_mpjpe_avg = train_mpjpe_total / len(train_dataloader)

    # VAL
    net.eval()
    val_loss_total = 0.0
    val_mpjpe_total = 0.0

    val_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_hat = net(x)
            loss = criterion(y_hat, y)

            val_loss_total += loss.item()
            val_mpjpe_total += mpjpe(y_hat, y).item()

    val_bar.close()
    val_loss_avg = val_loss_total / len(val_dataloader)
    val_mpjpe_avg = val_mpjpe_total / len(val_dataloader)

    scheduler.step()

    # EPOCH SUMMARY
    print(f"\n>>> Epoch {epoch+1}")
    print(f"Train Loss: {train_loss_avg:.5f} | Train MPJPE: {train_mpjpe_avg:.2f} mm")
    print(f"Val Loss:   {val_loss_avg:.5f} | Val MPJPE:   {val_mpjpe_avg:.2f} mm")
    print(f"Current LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_mpjpe_avg < best_val_mpjpe:
        best_val_mpjpe = val_mpjpe_avg
        torch.save(net.state_dict(), "best_hand_model.pth")
        print(f"New best model saved! (MPJPE: {best_val_mpjpe:.2f} mm)")
    print("\n")